# SolvER — train + evaluate Ditto on imputed / non-imputed datasets

In [ ]:
# ======================= EXPERIMENT SETTINGS =======================
DATASET          = "DBLPGoogleScholar"            # DBLPACM | DBLPScholar | walmartAmazon
PROJECT_ROOT = "/home/u59352/SolvIER2"
DATA_DIR         = "../Data"
SEEDS            = [1234, 584, 1576, 2534, 7430]               # seeds used when the null files were generated
NULL_PERCENTAGES = [5, 10, 20, 30, 40]      # cumulative % of nulls in the file names
MECHANISMS       = ["MCAR", "MAR", "MNAR"]   # missingness mechanism in the file names
VARIANTS         = ["nonimputed", "imputed_senti"]
EVAL_CLEAN       = True                 # also run the clean baseline (train/valid/test_<DATASET>.txt)
TRAIN_SEED = 0            # --run_id passed to train_ditto.py
N_EPOCHS   = 15
BATCH_SIZE = 8               # must be <= len(train set): num_steps = (len(train) // batch_size) * n_epochs
MAX_LEN    = 64
LR         = 3e-5
TRAIN_SIZE = None            # int -> subsample the train set, None -> use all
RETRAIN    = False           # True -> retrain even if checkpoints/<run>/model.pt exists

# --- Ditto's optional helpers (paper Sec 3). All OFF = the paper's "Baseline" row. ---
USE_DA          = False      # data augmentation (Sec 3.3). Also enables MixDA + entry_swap.
DA_OP           = "drop_col" # only used when USE_DA: del|swap|drop_col|append_col|drop_token|...
USE_DK          = None       # domain knowledge (Sec 3.1): None | "general" | "product"
USE_SUMMARIZE   = False      # TF-IDF summarization (Sec 3.2)
THRESHOLD_MODE  = "valid"    # where the decision threshold comes from:
THRESHOLD       = 0.5        # only used when THRESHOLD_MODE == "fixed"
DEBUG      = False           # True -> print train/matcher stdout/stderr

In [ ]:
import os, sys, re, json, subprocess, time, warnings
import pandas as pd
import torch
from tqdm import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)

# --- locate the project, wherever this kernel happened to start ---
def find_solver_dir():
    """The SolvER/ folder: PROJECT_ROOT, then $SOLVIER_ROOT, then walk up from cwd."""
    explicit = PROJECT_ROOT or os.environ.get("SOLVIER_ROOT")
    if explicit:
        d = os.path.join(os.path.abspath(os.path.expanduser(explicit)), "SolvIER", "SolvER")
        if not os.path.exists(os.path.join(d, "train_ditto.py")):
            raise FileNotFoundError(f"no SolvIER/SolvER/train_ditto.py under PROJECT_ROOT: {explicit}")
        return d
    here = os.path.abspath(os.getcwd())
    while True:
        for cand in (here, os.path.join(here, "SolvIER", "SolvER")):
            if os.path.exists(os.path.join(cand, "train_ditto.py")):
                return cand
        parent = os.path.dirname(here)
        if parent == here:
            raise FileNotFoundError(
                f"could not find SolvER/ from {os.getcwd()}.\n"
                f"Set PROJECT_ROOT in the settings cell to the SolvIER-main folder, "
                f"or export SOLVIER_ROOT=/path/to/SolvIER-main before starting Jupyter."
            )
        here = parent


SOLVER_DIR = find_solver_dir()

DATA_DIR    = os.path.abspath(os.path.join(SOLVER_DIR, DATA_DIR))  # relative paths resolve against SolvER/
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR does not exist: {DATA_DIR}  (set it in the settings cell)")
CONFIG_PATH = os.path.join(SOLVER_DIR, "configs.json")
OUTPUT_DIR  = os.path.join(SOLVER_DIR, "output")
CKPT_DIR    = os.path.join(SOLVER_DIR, "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)

USE_GPU = torch.cuda.is_available()
PYTHON  = sys.executable          # same interpreter as this kernel

print("dataset :", DATASET)
print("data    :", DATA_DIR)
print("gpu     :", USE_GPU, "" if USE_GPU else "(CPU only - training will be slow)")

## Helpers

In [ ]:
SPLITS = ("train", "valid", "test")

def experiment(mech=None, pct=None, seed=None, variant=None):
    """Describe one run: its name and its train/valid/test files.

    mech/pct/seed/variant = None -> clean baseline (<split>_<DATASET>.txt).
    Otherwise <split>_<DATASET>_<MECH>_<pct>_<seed>_<variant>.txt (.text also accepted),
    the naming the data-prep pipeline writes into ../Data/. `variant` is
    "nonimputed" or "imputed_senti".
    """
    suffix = DATASET if pct is None else f"{DATASET}_{mech}_{pct}_{seed}_{variant}"
    files = {}
    for split in SPLITS:
        # train/valid always come from the clean files - only the test file varies
        split_suffix = suffix if split == "test" else DATASET
        base = os.path.join(DATA_DIR, f"{split}_{split_suffix}")
        files[split] = next((base + ext for ext in (".txt", ".text") if os.path.exists(base + ext)), base + ".txt")
    return {"name": suffix,
            "model": DATASET,                                      # one checkpoint, shared by every experiment
            "mechanism": "clean" if mech is None else mech,
            "imputer": "clean" if variant is None else ("none" if variant == "nonimputed"
                                                        else variant.replace("imputed_", "")),
            "null_pct": 0 if pct is None else int(pct),
            "seed": -1 if seed is None else int(seed),
            "isImputed": -1 if variant is None else int(variant.startswith("imputed")),
            "files": files,
            "missing": [f for f in files.values() if not os.path.exists(f)]}


def update_config(exp):
    """Rewrite configs.json so train_ditto.py / matcher.py see this run's files."""
    config = [{
        "name": exp["model"],
        "task_type": "classification",
        "vocab": ["0", "1"],
        "trainset": exp["files"]["train"],
        "validset": exp["files"]["valid"],
        "testset":  exp["files"]["test"],
    }]
    with open(CONFIG_PATH, "w") as f:
        json.dump(config, f, indent=4)


def run(cmd):
    """Run a command in SOLVER_DIR, raise on failure."""
    if DEBUG:
        print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=SOLVER_DIR, capture_output=True, text=True)
    if DEBUG or proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")
    return proc.stdout

In [ ]:
TIMING = {"train_s": 0.0}   # training happens once; every row reports the same value


def train(exp):
    """Train Ditto on the clean train/valid files -> checkpoints/<DATASET>/model.pt.

    Called once per session: every experiment shares model name DATASET, so the
    `trained` set in the run loop short-circuits all later calls.
    """
    ckpt = os.path.join(CKPT_DIR, exp["model"], "model.pt")
    if not RETRAIN and os.path.exists(ckpt):
        print("  reusing checkpoint", ckpt)
        return
    update_config(exp)
    cmd = [PYTHON, "train_ditto.py",
           "--task", exp["model"],
           "--run_id", str(TRAIN_SEED),
           "--batch_size", str(BATCH_SIZE),
           "--max_len", str(MAX_LEN),
           "--lr", str(LR),
           "--n_epochs", str(N_EPOCHS),
           "--finetuning", "--lm", "distilbert",
           "--save_model",
           "--logdir", CKPT_DIR + "/"]
    if USE_DA:
        cmd += ["--da", DA_OP]          # augmentation + MixDA + entry_swap (training only)
    if USE_DK:
        cmd += ["--dk", USE_DK]
    if USE_SUMMARIZE:
        cmd += ["--summarize"]
    if TRAIN_SIZE:
        cmd += ["--size", str(TRAIN_SIZE)]
    t0 = time.perf_counter()
    out = run(cmd)
    TIMING["train_s"] = round(time.perf_counter() - t0, 1)
    epochs = [l for l in out.splitlines() if l.startswith("epoch")]
    print("  " + (epochs[-1] if epochs else "no epoch output"))
    print(f"  trained in {TIMING['train_s']:.1f}s")
    if not os.path.exists(ckpt):
        raise RuntimeError(f"training finished but {ckpt} was not written")


METRICS = ["load_f1", "real_f1", "TN", "TP", "FN", "FP", "FNR", "REC"]

def get_f1_values(s):
    """Parse the metrics printed by matcher.py; -1 when a metric is missing."""
    vals = []
    for m in METRICS:
        hit = re.search(rf"{m}\s*=\s*([0-9.eE+-]+)", s)
        vals.append(float(hit.group(1)) if hit else -1)
    return vals


def evaluate(exp):
    """Run matcher.py with exp's model on exp's test file; return one result row.

    `match_s` is wall-clock for the whole matcher.py call on this file, so it
    includes the one-off model load - that is what "time per file" costs in practice.
    `ms_per_pair` divides it by the pairs in the file, for a size-independent figure.
    """
    update_config(exp)
    cmd = [PYTHON, "matcher.py",
           "--task", exp["model"],
           "--input_path", exp["files"]["test"],
           "--output_path", os.path.join(OUTPUT_DIR, f"output_{exp['name']}.jsonl"),
           "--lm", "distilbert",
           "--max_len", str(MAX_LEN),
           "--threshold_mode", THRESHOLD_MODE,
           "--threshold", str(THRESHOLD),
           "--checkpoint_path", CKPT_DIR + "/"]
    if USE_DK:
        cmd += ["--dk", USE_DK]
    if USE_SUMMARIZE:
        cmd.append("--summarize")
    if USE_GPU:
        cmd.append("--use_gpu")
    t0 = time.perf_counter()
    out = run(cmd)
    match_s = round(time.perf_counter() - t0, 1)      # this file only: model load + threshold + predict
    load_f1, real_f1, tn, tp, fn, fp, fnr, rec = get_f1_values(out)
    pairs = sum(1 for l in open(exp["files"]["test"], encoding="utf-8") if l.strip())
    print(f"  real_f1 = {real_f1:.4f}   match_s = {match_s:.1f}  ({pairs} pairs)")
    return (exp["name"], exp["mechanism"], exp["imputer"], exp["null_pct"], exp["seed"],
            exp["isImputed"], load_f1, real_f1, tn, tp, fn, fp, fnr, rec,
            pairs, match_s, round(1000 * match_s / pairs, 2) if pairs else -1,
            TIMING["train_s"])

## Run all experiments

In [ ]:
# build the experiment list from the settings
experiments = [experiment()] if EVAL_CLEAN else []
for seed in SEEDS:
    for mech in MECHANISMS:
        for pct in NULL_PERCENTAGES:
            for variant in VARIANTS:
                experiments.append(experiment(mech, pct, seed, variant))

runnable = [e for e in experiments if not e["missing"]]
skipped  = [e for e in experiments if e["missing"]]
if skipped:
    print(f"WARNING: {len(skipped)} experiment(s) skipped, missing files:")
    for e in skipped:
        print(f"   {e['name']}: " + ", ".join(os.path.basename(f) for f in e["missing"]))
if not runnable:
    raise FileNotFoundError(f"No complete train/valid/test triple found in {DATA_DIR} for DATASET={DATASET}, SEEDS={SEEDS}, MECHANISMS={MECHANISMS}, NULL_PERCENTAGES={NULL_PERCENTAGES}")
print(f"{len(runnable)} experiment(s) to run\n")

results = []
trained = set()                       # all runs share one clean-trained model: train it once
for i, exp in enumerate(runnable, 1):
    print(f"[{i}/{len(runnable)}] {exp['name']}  (model: {exp['model']})")
    if exp["model"] not in trained:
        train(exp)
        trained.add(exp["model"])
    results.append(evaluate(exp))

TEXT_COLUMNS = ["run", "mechanism", "imputer"]
NUM_COLUMNS = ["null_pct", "seed", "isImputed", "load_f1", "real_f1", "tn", "tp", "fn", "fp", "fnr", "rec",
               "pairs", "match_s", "ms_per_pair", "train_s"]
COLUMNS = TEXT_COLUMNS + NUM_COLUMNS
df = pd.DataFrame(results, columns=COLUMNS)


def from_run(run):
    """(mechanism, imputer) read back off a run name, for rows written by older versions."""
    parts = str(run).split("_")
    if len(parts) < 5:
        return "clean", "clean"
    variant = "_".join(parts[4:])
    if variant == "nonimputed":
        return parts[1], "none"
    if variant == "imputed":
        return parts[1], "senti"      # rows from before the strategy was in the file name
    return parts[1], variant.replace("imputed_", "")


# append to the master CSV; a rerun of the same experiment replaces its old row
RESULTS_CSV = os.path.join(OUTPUT_DIR, f"Results_{DATASET}_{SEEDS}.csv")
if os.path.exists(RESULTS_CSV):
    df = pd.concat([pd.read_csv(RESULTS_CSV).reindex(columns=COLUMNS), df], ignore_index=True)
for col in NUM_COLUMNS:                      # text columns stay text - to_numeric would blank them
    df[col] = pd.to_numeric(df[col], errors="coerce")
blank = df["mechanism"].isna() | df["imputer"].isna()
if blank.any():
    df.loc[blank, ["mechanism", "imputer"]] = [from_run(r) for r in df.loc[blank, "run"]]
df = (df.dropna(subset=["run", "seed", "null_pct"])
        .astype({"null_pct": int, "seed": int, "isImputed": int})
        .drop_duplicates(subset=["run"], keep="last")
        .sort_values(["mechanism", "imputer", "seed", "null_pct", "isImputed"])
        .reset_index(drop=True))
df.to_csv(RESULTS_CSV, index=False)
print("saved", RESULTS_CSV)
if "match_s" in df and df.match_s.notna().any():
    done = df[df.match_s.notna()]
    print(f"matching time: {done.match_s.sum():.0f}s total over {len(done)} files, "
          f"{done.match_s.mean():.1f}s per file, {done.ms_per_pair.mean():.2f} ms per pair"
          + (f"; training {TIMING['train_s']:.0f}s" if TIMING["train_s"] else ""))
df

## Summary: mean F1 per mechanism and null % — imputed (1) vs non-imputed (0), averaged over seeds

In [ ]:
summary = (df[df.seed >= 0]
           .groupby(["mechanism", "null_pct", "isImputed"])["real_f1"]
           .agg(["mean", "std", "count"])
           .unstack("isImputed"))
summary

# 